In [1]:
# (supply‑risk metrics, HHI, sector usage)
"""
Notebook scope— Transform
supply concentration (HHI)

producer diversity

sector usage counts

economic importance
"""

# %%
import pandas as pd

from utils.utility import (
    save_parquet, load_parquet,
)

In [2]:
# load all data
matrix = load_parquet("mineral_country_matrix.parquet")
country_totals = load_parquet("country_mineral_counts.parquet")
mineral_totals = load_parquet("global_mineral_counts.parquet")

mineral_sector = load_parquet("mineral_sector_mat.parquet")
sector_country = load_parquet("sector_country_matrix.parquet")
sector_elasticity = load_parquet("sector_elasticity.parquet")

sector_codes = load_parquet("sector_codes.parquet")
mineral_codes = load_parquet("mineral_codes.parquet")
country_codes = load_parquet("country_codes.parquet")

country_master  = load_parquet("country_master.parquet")

In [3]:
#1. Supply‑side metrics
producer_totals = mineral_totals
producer_totals = mineral_totals.set_index("Mineral Code")["Mineral Counts"]

# HHI = sum of squared shares
"""The Herfindahl-Hirschman Index (HHI) measures market concentration by summing the squares of the market shares of all firms in an industry, indicating the level of competition.
low HHI = low supply risk
high HHI = high supply risk
"""
# share = 1 / producer_count
hhi = (1 / producer_totals).rename("HHI")
producer_diversity = producer_totals.rename("Producer Diversity")


In [4]:
#2. Demand‑side metrics

# Sector usage count per mineral
sector_usage = mineral_sector.sum(axis=0).rename("Sector Usage Count")

sector_usage_df = (
    sector_usage
    .to_frame()
    .reset_index()
    .rename(columns={"index": "Mineral Code"})
)

save_parquet(sector_usage_df, "sector_usage_counts.parquet")

# Load processed data
mineral_sector = load_parquet("mineral_sector_mat.parquet")
sector_elasticity = load_parquet("sector_elasticity.parquet")

# Build elasticity lookup
# Normalise sector codes in both tables
mineral_sector.index = mineral_sector.index.str.strip().str.upper()
sector_elasticity["Sector Code"] = sector_elasticity["Sector Code"].str.strip().str.upper()
# Rebuild elasticity map
elasticity_map = sector_elasticity.set_index("Sector Code")["Elasticity"]
# Weight each sector row by (1 - elasticity)
weighted_sector = mineral_sector.mul(1 - elasticity_map, axis=0)

# Sum across sectors to get EI per mineral
economic_importance = (
    weighted_sector.sum(axis=0)
    .rename("Economic Importance")
)
df_ei = (
    economic_importance
    .rename("Economic Importance")
    .reset_index()
    .rename(columns={"index": "Mineral Code"})
)


save_parquet(df_ei, "economic_importance.parquet")



In [5]:
transform = pd.concat(
    [
        producer_totals.rename("Mineral Counts"),
        hhi,
        producer_diversity,
        sector_usage,
        economic_importance
    ],
    axis=1
).reset_index().rename(columns={"index": "Mineral Code"})

save_parquet(transform, "notepad_02_transform_metrics.parquet")

In [7]:
# -----------------------------
# LOAD MASTER DATASET
# -----------------------------
master = country_master.copy()

master["Producer Count"] = master["Producer Count"].fillna(0)
master["Manufacturer Count"] = master["Manufacturer Count"].fillna(0)

# -----------------------------
# PRODUCER & MANUFACTURER STRENGTH
# -----------------------------
master["Producer Strength"] = master["Producer Count"] / master["Producer Count"].max()
master["Manufacturer Strength"] = master["Manufacturer Count"] / master["Manufacturer Count"].max()

# -----------------------------
# LINK SCORE
# -----------------------------
master["Link Score"] = master["Producer Strength"] * master["Manufacturer Strength"]

# -----------------------------
# GAP & ASYMMETRY
# -----------------------------
master["Gap"] = master["Producer Strength"] - master["Manufacturer Strength"]
master["Asymmetry Index"] = master["Gap"].abs()

# -----------------------------
# CAPABILITY SCORE
# -----------------------------
master["GDP_norm"] = master["GDP_IMF_2026"] / master["GDP_IMF_2026"].max()
master["Land_norm"] = master["Land_Area_km2"] / master["Land_Area_km2"].max()
master["Pop_norm"] = master["Population"] / master["Population"].max()
master["Ineq_penalty"] = 1 - (master["GNI_Coefficient"] / master["GNI_Coefficient"].max())

master["Capability Score"] = (
    0.25 * master["Producer Strength"] +
    0.25 * master["Manufacturer Strength"] +
    0.20 * master["GDP_norm"] +
    0.10 * master["Land_norm"] +
    0.10 * master["Pop_norm"] +
    0.10 * master["Ineq_penalty"]
)

# -----------------------------
# DEPENDENCY CLASSIFICATION
# -----------------------------
def classify(row):
    p, m = row["Producer Count"], row["Manufacturer Count"]
    if p > 0 and m > 0: return "Integrated"
    if p > 0 and m == 0: return "Producer Only"
    if p == 0 and m > 0: return "Manufacturer Only"
    return "Neither"

master["Dependency Class"] = master.apply(classify, axis=1)

# -----------------------------
# EXPORT NOTEBOOK 2 OUTPUTS
# -----------------------------
save_parquet(master, "country_links.parquet")


In [8]:
groups = master.groupby("Dependency Class")
summary = groups[["GDP_IMF_2026", "Population", "Land_Area_km2", "GNI_Coefficient"]].describe()


In [10]:
print(summary)

                  GDP_IMF_2026                                        \
                         count          mean           std       min   
Dependency Class                                                       
Integrated                13.0  6.184761e+06  9.484796e+06  194333.0   
Manufacturer Only         10.0  1.743743e+06  1.378223e+06  110417.0   
Neither                  135.0  1.420448e+09  3.242461e+09    1143.0   
Producer Only             38.0  3.258248e+09  1.754649e+10   17314.0   

                                                                     \
                          25%        50%          75%           max   
Dependency Class                                                      
Integrated         1640223.00  2507340.0    4379253.0  3.238392e+07   
Manufacturer Only   784065.75  1298307.5    2536375.0  4.264794e+06   
Neither              35634.00   121537.0  618805000.0  1.999344e+10   
Producer Only        32758.25   203419.5     523505.0  1.073520e+11  